In [1]:
import pandas as pd
from collections import Counter

print("=" * 80)
print("Investigating Unknown Document with 158 Selections")
print("=" * 80)
print()

Investigating Unknown Document with 158 Selections



In [3]:
# 读取数据
try:
    logs_df = pd.read_csv('logs.csv')
    print(f"✓ Logs loaded: {len(logs_df)} records")
except:
    try:
        # 如果没有 CSV，尝试从 SQL 解析
        import re
        
        def parse_sql_insert_statements(sql_file_path, table_name='logs'):
            print(f"Parsing SQL file: {sql_file_path}")
            data = []
            with open(sql_file_path, 'r', encoding='utf-8') as f:
                buffer = ""
                insert_pattern = f"INSERT INTO `{table_name}` VALUES "
                for line_num, line in enumerate(f, 1):
                    if line_num % 100000 == 0:
                        print(f"  Processed {line_num} lines...")
                    if line.startswith('--') or line.startswith('/*') or line.startswith('*/') or not line.strip():
                        continue
                    buffer += line
                    if insert_pattern in buffer:
                        try:
                            values_start = buffer.find(insert_pattern) + len(insert_pattern)
                            values_section = buffer[values_start:]
                            pattern = r'\(([^)]+)\)'
                            matches = re.findall(pattern, values_section)
                            for match in matches:
                                values = []
                                current = ""
                                in_quotes = False
                                quote_char = None
                                for char in match:
                                    if char in ('"', "'") and (not in_quotes or char == quote_char):
                                        if in_quotes and char == quote_char:
                                            in_quotes = False
                                            quote_char = None
                                        elif not in_quotes:
                                            in_quotes = True
                                            quote_char = char
                                        current += char
                                    elif char == ',' and not in_quotes:
                                        values.append(current.strip().strip('"').strip("'"))
                                        current = ""
                                    else:
                                        current += char
                                if current:
                                    values.append(current.strip().strip('"').strip("'"))
                                if len(values) == 10:
                                    data.append(values)
                            buffer = ""
                        except Exception as e:
                            buffer = ""
                            continue
            columns = ['id', 'user_id', 'qid', 'docno', 'event_type', 
                       'start_idx', 'end_idx', 'duration', 'pass_flag', 'timestamp']
            df = pd.DataFrame(data, columns=columns)
            df['id'] = pd.to_numeric(df['id'], errors='coerce')
            df['qid'] = pd.to_numeric(df['qid'], errors='coerce')
            df['start_idx'] = pd.to_numeric(df['start_idx'], errors='coerce')
            df['end_idx'] = pd.to_numeric(df['end_idx'], errors='coerce')
            df['duration'] = pd.to_numeric(df['duration'], errors='coerce')
            df['pass_flag'] = pd.to_numeric(df['pass_flag'], errors='coerce')
            return df
        
        logs_df = parse_sql_insert_statements('backup_qe_first_half.sql')
        print(f"✓ Logs parsed from SQL: {len(logs_df)} records")
    except:
        print("✗ Cannot load data. Please ensure logs.csv or SQL file exists.")
        exit(1)

try:
    interleaving_df = pd.read_csv('result_interleaved_with_text.csv')
    print(f"✓ Interleaving results loaded: {len(interleaving_df)} records")
except:
    print("✗ interleaving_results.csv not found.")
    exit(1)

print()

Parsing SQL file: backup_qe_first_half.sql
✓ Logs parsed from SQL: 27466 records
✓ Interleaving results loaded: 387 records



In [4]:
# ============================================================================
# 第二步：筛选目标用户
# ============================================================================

print("=" * 80)
print("User Filtering")
print("=" * 80)
print()

# 方法1：手动指定27个用户ID（如果你已经知道）
MANUAL_USERS = ['5b68c9eb87af310001584803',
                '55d51a6b8ce09000127d4821',
                '67181a55422d48f5727f08ec',
                '595cd88562431800019d691a',
                '610c66163e803a3564dfaf9c',
                '66435e2e5c5b5aceb2195908',
                '628f572a2d1304a34b9e1a37',
                '65de30b34b701c678a13c75b',
                '5f540fc665e3b6148c04f10e',
                '5f1088e27a43fe0c64ca5814',
                '66cc6eb364154a51d4d44eb1',
                '66cec5a3fdf1fe2c010e9971',
                '5c6e60b04e08ad00018cc995',
                '673e222bdb1a35935b3075f1',
                '659c067de92634488b349626',
                '5d68c8aa40524c00189e8ac2',
                '66fa1820329fa9bb88f2e3c6',
                '6012e262527c380ccb2a6c74',
                '668a9e53e7690b3b64d48e92',
                '5ca102f57114700016da340d',
                '5c6709ab926a5b0001eb29c1',
                '66d9cd06e207a93377f7820e',
                '6151e479b4ee16668948c7f2',
                '5d542c3a5af5900019bc80c4',
                '5f565a2b33c152071057bb04',
                '62d6d77df59450e545fb22f6',
                '63cedb764240865b7177c745']

# 方法2：自动选择（如果列表为空，设置为[]使用所有用户）
if len(MANUAL_USERS) == 0:
    print("No manual user list provided. Using all users in the dataset.")
    target_users = logs_df['user_id'].unique().tolist()
else:
    target_users = MANUAL_USERS
    print(f"Using manually specified user list: {len(target_users)} users")

print(f"Target users to analyze: {len(target_users)}")
print()

# 筛选目标用户的日志
print("Filtering logs for target users...")
original_count = len(logs_df)
logs_df = logs_df[logs_df['user_id'].isin(target_users)].copy()
filtered_count = len(logs_df)

print(f"✓ Filtered from {original_count} to {filtered_count} records")
print(f"✓ Logs from {logs_df['user_id'].nunique()} unique users")

if logs_df['user_id'].nunique() < len(target_users):
    missing_users = set(target_users) - set(logs_df['user_id'].unique())
    print(f"⚠️  Warning: {len(missing_users)} users have no logs in the dataset")
    if len(missing_users) <= 5:
        print(f"   Missing users: {list(missing_users)[:5]}")

print()

User Filtering

Using manually specified user list: 27 users
Target users to analyze: 27

Filtering logs for target users...
✓ Filtered from 27466 to 7082 records
✓ Logs from 27 unique users



In [5]:
# 创建文档映射
doc_label_map = {}
for _, row in interleaving_df.iterrows():
    key = (str(row['qid']), str(row['docno']))
    doc_label_map[key] = row['origin_label']

print(f"Document mapping created: {len(doc_label_map)} documents")
print()

# 筛选 PASSAGE_SELECTION 事件
passage_logs = logs_df[logs_df['event_type'] == 'PASSAGE_SELECTION'].copy()
print(f"Total PASSAGE_SELECTION events: {len(passage_logs)}")
print()

# 找出所有 Unknown 文档
unknown_docs = []
for _, log in passage_logs.iterrows():
    qid_str = str(int(log['qid'])) if pd.notna(log['qid']) else '0'
    docno_str = str(log['docno'])
    key = (qid_str, docno_str)
    
    if key not in doc_label_map:
        unknown_docs.append({
            'qid': qid_str,
            'docno': docno_str,
            'user_id': log['user_id']
        })

print(f"Found {len(unknown_docs)} Unknown document selections")
print()

# 统计每个 Unknown 文档的选择次数
if len(unknown_docs) > 0:
    unknown_doc_keys = [(d['qid'], d['docno']) for d in unknown_docs]
    doc_counts = Counter(unknown_doc_keys)
    
    print("=" * 80)
    print("Unknown Document Selection Counts")
    print("=" * 80)
    print()
    
    # 按选择次数排序
    sorted_docs = sorted(doc_counts.items(), key=lambda x: x[1], reverse=True)
    
    for (qid, docno), count in sorted_docs:
        print(f"QID: {qid}, DocNo: {docno} → Selected {count} times")
    
    print()
    
    # 重点分析被选择最多的文档
    top_doc = sorted_docs[0]
    top_qid, top_docno = top_doc[0]
    top_count = top_doc[1]
    
    print("=" * 80)
    print(f"Detailed Analysis: Most Selected Unknown Document")
    print("=" * 80)
    print()
    
    print(f"QID: {top_qid}")
    print(f"DocNo: {top_docno}")
    print(f"Total Selections: {top_count}")
    print()
    
    # 找出所有选择这个文档的用户
    users_selected = []
    for d in unknown_docs:
        if d['qid'] == top_qid and d['docno'] == top_docno:
            users_selected.append(d['user_id'])
    
    unique_users = set(users_selected)
    print(f"Number of unique users who selected this document: {len(unique_users)}")
    print()
    
    # 显示这些用户
    print("Users who selected this document:")
    for i, user in enumerate(unique_users, 1):
        user_selections = users_selected.count(user)
        print(f"  {i}. User {user}: {user_selections} times")
    
    print()
    
    # 检查这个查询在 interleaving_results 中的情况
    print("=" * 80)
    print(f"Query {top_qid} in interleaving_results")
    print("=" * 80)
    print()
    
    query_docs = interleaving_df[interleaving_df['qid'].astype(str) == top_qid]
    
    if len(query_docs) == 0:
        print(f"⚠️  WARNING: Query {top_qid} does NOT exist in interleaving_results!")
        print("This explains why all documents from this query are Unknown.")
        print()
        print("Possible reasons:")
        print("  1. This query was not included in the interleaving experiment")
        print("  2. interleaving_results.csv is incomplete")
        print("  3. This is a different experiment or pilot study")
    else:
        print(f"Query {top_qid} has {len(query_docs)} documents in interleaving_results")
        print()
        print("Documents for this query:")
        print(query_docs[['qid', 'docno', 'origin_label', 'rank']].to_string(index=False))
        print()
        
        # 检查我们的 Unknown 文档是否在列表中
        if top_docno in query_docs['docno'].astype(str).values:
            print(f"✓ DocNo {top_docno} EXISTS in interleaving_results for this query")
            print("This is a data type mismatch issue!")
            
            matching_doc = query_docs[query_docs['docno'].astype(str) == top_docno].iloc[0]
            print()
            print("Matching document details:")
            print(f"  QID: {matching_doc['qid']} (type: {type(matching_doc['qid'])})")
            print(f"  DocNo: {matching_doc['docno']} (type: {type(matching_doc['docno'])})")
            print(f"  Origin Label: {matching_doc['origin_label']}")
        else:
            print(f"✗ DocNo {top_docno} NOT FOUND in interleaving_results for this query")
            print()
            print(f"Available DocNos for query {top_qid}:")
            for docno in query_docs['docno'].head(10):
                print(f"  - {docno}")
    
    print()
    
    # 分析这个查询的所有 passage selections
    print("=" * 80)
    print(f"All Passage Selections for Query {top_qid}")
    print("=" * 80)
    print()
    
    query_passages = passage_logs[passage_logs['qid'].astype(str) == top_qid]
    print(f"Total passages for this query: {len(query_passages)}")
    print()
    
    # 统计这个查询的文档选择
    query_doc_counts = query_passages.groupby('docno').size().sort_values(ascending=False)
    print("Document selection counts for this query:")
    for docno, count in query_doc_counts.items():
        # 检查是否是 Unknown
        key = (top_qid, str(docno))
        label = doc_label_map.get(key, 'Unknown')
        marker = " ⚠️ " if label == 'Unknown' else ""
        print(f"  DocNo {docno}: {count} selections ({label}){marker}")
    
    print()

Document mapping created: 387 documents

Total PASSAGE_SELECTION events: 2700

Found 158 Unknown document selections

Unknown Document Selection Counts

QID: 2082, DocNo: msmarco_passage_45_623131157 → Selected 2 times
QID: 30611, DocNo: msmarco_passage_01_436571677 → Selected 2 times
QID: 112700, DocNo: msmarco_passage_30_794527402 → Selected 2 times
QID: 2082, DocNo: msmarco_passage_30_709623997 → Selected 1 times
QID: 2082, DocNo: msmarco_passage_64_431617227 → Selected 1 times
QID: 2082, DocNo: msmarco_passage_45_632929045 → Selected 1 times
QID: 2082, DocNo: msmarco_passage_55_738286358 → Selected 1 times
QID: 2082, DocNo: msmarco_passage_26_846132892 → Selected 1 times
QID: 2082, DocNo: msmarco_passage_40_137452930 → Selected 1 times
QID: 2082, DocNo: msmarco_passage_39_461879190 → Selected 1 times
QID: 23287, DocNo: msmarco_passage_37_406984666 → Selected 1 times
QID: 23287, DocNo: msmarco_passage_03_865272429 → Selected 1 times
QID: 23287, DocNo: msmarco_passage_26_607486542 → 